In [1]:
import os
from datetime import datetime
import pandas as pd
import yfinance as yf

OUTPUT_DIR = "Incoming_Data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def fetch_market_data():
    tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
    print("Fetching latest financial data from Yahoo Finance...")
    
    processed_dfs = []
    for ticker in tickers:
        try:
            print(f"Downloading {ticker}...")
            # Download one ticker at a time sequentially to avoid database locking
            df = yf.download(ticker, period="1y", auto_adjust=True)
            
            if not df.empty:
                # Flatten multi-index columns if present in newer yfinance versions
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                
                df["Ticker"] = ticker
                df = df.reset_index()
                processed_dfs.append(df)
        except Exception as e:
            print(f"Error processing {ticker}: {e}")
            
    if processed_dfs:
        master_df = pd.concat(processed_dfs, ignore_index=True)
        # Ensure correct column selection
        cols_to_use = [col for col in ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"] if col in master_df.columns]
        master_df = master_df[cols_to_use].dropna()
        
        file_name = f"market_data_snapshot_{datetime.now().strftime('%Y-%m-%d')}.csv"
        file_path = os.path.join(OUTPUT_DIR, file_name)
        
        master_df.to_csv(file_path, index=False)
        print(f"Success! Saved to: {file_path}")
    else:
        print("No data was successfully fetched.")

if __name__ == "__main__":
    fetch_market_data()

Fetching latest financial data from Yahoo Finance...


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed

Success! Saved to: Incoming_Data\market_data_snapshot_2026-11-18.csv
